# Bird flu in China, 2013: an epicurve

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leehart3000/epiplot/blob/main/docs/case-studies/h7n9-china-2013.ipynb)

In spring 2013, a new bird flu virus, influenza A(H7N9), began infecting people
in eastern China. Most people who fell ill had been in contact with live poultry,
often at markets. This case study draws an **epidemic curve** (epicurve) of the
first wave, using a public line list of 136 cases.

An epicurve answers the first questions about any outbreak: *when did people
fall ill, and how many?* Its shape gives clues about how the disease is spreading
and whether control measures are working.

## Set up

If you are running this notebook in Google Colab, the next cell installs
**epiplot** first. Elsewhere it does nothing.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet epiplot

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import epiplot

## Load the line list

A **line list** has one row per case. This one was collated by Kucharski and
colleagues from WHO reports, ProMED, FluTrackers and news reports. Our copy keeps
only the columns needed here: see the data notes at the end.

In [ ]:
DATA = Path("data/h7n9_china_2013.csv")
if not DATA.exists():  # for example in Colab: read the copy on GitHub instead
    DATA = (
        "https://raw.githubusercontent.com/leehart3000/epiplot/main/"
        "docs/case-studies/data/h7n9_china_2013.csv"
    )

cases = pd.read_csv(DATA)
cases.head()

Real data are rarely complete. It is worth checking how many values are missing
before plotting:

In [ ]:
cases.isna().sum()

Ten cases have no date of symptom onset. They can't be placed on the curve, but
they shouldn't silently disappear either: **epiplot** will say how many were left
out, under the plot.

Most cases were in three places: the provinces of Zhejiang and Jiangsu, and the
city of Shanghai (which is counted as a province here).

In [ ]:
cases["province"].value_counts().head(6)

## Group the provinces

Colours are easiest to tell apart when there are only a few. So we keep the three
most affected areas and combine the rest as "Other". Making the groups a
*categorical* column fixes their order: the first group sits at the bottom of each
bar, on the flat baseline, where its rise and fall is easiest to read.

In [ ]:
top3 = ["Zhejiang", "Shanghai", "Jiangsu"]
cases["province_group"] = pd.Categorical(
    cases["province"].where(cases["province"].isin(top3), "Other"),
    categories=[*top3, "Other"],
)

## Draw the epicurve

In [ ]:
ax = epiplot.epicurve(
    cases,
    "onset_date",
    date_type="onset",
    group_col="province_group",
)
ax.set_title("Human cases of influenza A(H7N9), China, 2013")
plt.show()

## What the curve shows

- **A sharp spring wave.** Cases rose quickly through March, peaked in the week
  starting 8 April, then fell away by early May.
- **The decline came soon after action at the source.** In early April, several
  affected cities closed their live poultry markets. A later study estimated
  that these closures greatly reduced the risk of infection (Yu et al., 2014).
  An epicurve alone can't prove cause and effect, but it is where such questions
  start.
- **A few later cases.** Scattered cases in May and July show the virus had not
  gone away, which matched what happened next: H7N9 returned in later winters.

## Good practice, built in

Notice what **epiplot** did without being asked:

- **The x axis says what the dates mean:** "Week of symptom onset". Onset and
  report dates tell different stories, so the axis label must say which is used.
- **Empty weeks are shown as gaps at zero,** so the quiet weeks between the main
  wave and the later cases stay visible.
- **The 10 cases without an onset date are reported** under the plot, rather than
  quietly dropped.
- **Colours are colour-blind-safe,** and the week start (Monday) is stated.

## Try it yourself

- Group by `outcome` instead of province, to see whether deaths cluster in time.
- Use `interval="day"` to see the same outbreak in more detail.
- Call `epiplot.count_cases(...)` with the same options to get the numbers behind
  the plot as a table.

## Data source and licence

Kucharski A, Mills HL, Pinsent A, Fraser C, Van Kerkhove M, Donnelly CA, Riley S
(2015). Data from: Distinguishing between reservoir exposure and human-to-human
transmission for emerging pathogens using case onset data. Dryad.
<https://doi.org/10.5061/dryad.2g43n>. Released under CC0 1.0.

Our copy keeps only the case ID, sex, province, onset date and outcome. Free-text
notes, ages and other details about individual patients were left out. The
script that made our copy, and notes on every change, are in the
[data folder on GitHub](https://github.com/leehart3000/epiplot/tree/main/docs/case-studies/data).

Yu H, Wu JT, Cowling BJ, et al. (2014). Effect of closure of live poultry markets
on poultry-to-person transmission of avian influenza A H7N9 virus: an ecological
study. *The Lancet* 383(9916): 541–548.